
Mount G-drive

In [ ]:
# ── Install ────────────────────────────────────────────────
!pip install transformers torch h5py tqdm -q

# ── Mount Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:


import os, torch, numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

# ── Config ─────────────────────────────────────────────────
# INPUT_FILE  = "/content/drive/MyDrive/wiki-samples/wiki_0_sentences.txt"   # ← your source text file
# OUTPUT_DIR  = "/content/drive/MyDrive/wiki_embeddings"

# Replace with:
INPUT_FILE = "/content/text.txt"
# for multiple files
# INPUT_FILES = [
#     "./data/wiki_0_sentences.txt",
#     "./data/wiki_1_sentences.txt",
#     "./data/wiki_2_sentences.txt",
# ]  # ← path to their local .txt file
OUTPUT_DIR  = "./wiki_embeddings" # path to save embeddings
MAX_SEQ_LEN = 512
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = [
    ("bert_tiny",  "prajjwal1/bert-tiny",  128, 512),
    ("bert_mini",  "prajjwal1/bert-mini",  256, 512),
    ("bert_small", "prajjwal1/bert-small", 512, 256),
    ("bert_base",  "bert-base-uncased",    768, 128),
    # (save_name, hf_model_id, embed_dim, batch_size)
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Helpers ────────────────────────────────────────────────
def read_all_sentences(filepath):
    """Read and return all non-empty lines from the text file."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        return [line.strip() for line in f if line.strip()]

@torch.no_grad()
def encode_cls(model, tokenizer, sentences):
    """CLS-token pooling: returns (batch_size, hidden_dim) float32 array."""
    enc = tokenizer(
        sentences, padding=True, truncation=True,
        max_length=MAX_SEQ_LEN, return_tensors="pt"
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = model(**enc)
    return out.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32)

# ── Load sentences once ────────────────────────────────────
print(f"Reading sentences from:\n  {INPUT_FILE}\n")
sentences = read_all_sentences(INPUT_FILE)
print(f"Total sentences loaded: {len(sentences):,}\n")

# Optionally save the full sentence list once (shared across models)
sentences_path = os.path.join(OUTPUT_DIR, "sentences_all.txt")
with open(sentences_path, "w", encoding="utf-8") as f:
    f.write("\n".join(sentences))
print(f"Sentences saved → {sentences_path}\n")

# ── Generate embeddings for each model ────────────────────
for model_name, hf_id, embed_dim, batch_size in MODELS:
    print(f"{'='*60}")
    print(f"Model  : {model_name}  ({hf_id})")
    print(f"Dim    : {embed_dim}  |  Batch size: {batch_size}  |  Device: {DEVICE}")

    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model     = AutoModel.from_pretrained(hf_id).to(DEVICE).eval()

    all_embeddings = []
    batches = range(0, len(sentences), batch_size)

    for i in tqdm(batches, desc=f"  Encoding [{model_name}]"):
        batch = sentences[i : i + batch_size]
        embs  = encode_cls(model, tokenizer, batch)
        all_embeddings.append(embs)

    # Concatenate all batch outputs → (N, embed_dim)
    merged = np.concatenate(all_embeddings, axis=0)
    tensor = torch.from_numpy(merged).half()   # store as fp16 to halve disk usage

    save_path = os.path.join(OUTPUT_DIR, f"embeddings_{model_name}.pt")
    torch.save(tensor, save_path)

    file_mb = os.path.getsize(save_path) / 1e6
    print(f"  ✓ Saved  shape={tuple(tensor.shape)}  →  {save_path}  ({file_mb:.1f} MB)\n")

    # Free GPU/CPU memory before loading the next model
    del model, tokenizer, all_embeddings, merged, tensor
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("✅  All models done.")

Reading sentences from:
  /content/text.txt

Total sentences loaded: 4

Sentences saved → ./wiki_embeddings/sentences_all.txt

Model  : bert_tiny  (prajjwal1/bert-tiny)
Dim    : 128  |  Batch size: 512  |  Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Encoding [bert_base]: 100%|██████████| 1/1 [00:00<00:00, 35.00it/s]

  ✓ Saved  shape=(4, 768)  →  ./wiki_embeddings/embeddings_bert_base.pt  (0.0 MB)

✅  All models done.


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
import os
import copy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import normalized_mutual_info_score
from scipy.stats import spearmanr


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

COMPRESS_DIMS = [128, 256, 512]

SMALL_MODELS = [
    ("bert_tiny",  128),
    ("bert_mini",  256),
    ("bert_small", 512),
]

# NEW: Multiple k values for kNN evaluation
# Changed K_VALUES to be compatible with N=4 sentences (max k = N-1 = 3)
K_VALUES = [5, 10, 20]

CONFIG = {
    'input_dim':       768,
    'epochs':          40,
    'batch_size':      2048,
    'lr':              1e-3,
    'tau':             0.05,
    'dropout_rate':    0.1,
    'moco_queue_size': 65536,
}

# Evaluation settings
N_CLUSTERS         = 20      # for clustering NMI evaluation
STS_SAMPLE_SIZE    = 5000    # sample size for cosine sim correlation
STS_N_PAIRS        = 50_000  # number of random pairs for correlation
LATENCY_N_QUERIES  = 200     # queries to average for latency benchmark

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ─────────────────────────────────────────────────────────────────────────────
# 1. ADAPTIVE ENCODER
# ─────────────────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, input_dim=768, latent_dim=128, dropout_rate=0.1):
        super().__init__()
        dims = [input_dim]
        d = input_dim

        while d > latent_dim * 2:
            d = max(d // 2, latent_dim * 2)
            dims.append(d)

        # The fix for the 512D shape mismatch
        if len(dims) == 1 or (len(dims) == 2 and dims[1] == latent_dim):
            dims.append(max(input_dim // 2, latent_dim + 1))

        dims.append(latent_dim)

        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers += [nn.LayerNorm(dims[i + 1]), nn.GELU(), nn.Dropout(dropout_rate)]
        self.net = nn.Sequential(*layers)

    # Restored: PyTorch needs this to push tensors through the network!
    def forward(self, x):
        return self.net(x)

# ─────────────────────────────────────────────────────────────────────────────
# 2. MoCo QUEUE
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def momentum_update(model_q, model_k, m=0.999):
    for pq, pk in zip(model_q.parameters(), model_k.parameters()):
        pk.data = pk.data * m + pq.data * (1.0 - m)


class MoCoQueue:
    def __init__(self, size, dim, device):
        self.size   = size
        self.ptr    = 0
        self.buffer = F.normalize(torch.randn(size, dim, device=device), dim=1)

    def get(self):
        return self.buffer.detach()

    @torch.no_grad()
    def enqueue(self, z_norm):
        B = z_norm.size(0)
        end = self.ptr + B
        if end <= self.size:
            self.buffer[self.ptr:end] = z_norm
        else:
            tail = self.size - self.ptr
            self.buffer[self.ptr:]        = z_norm[:tail]
            self.buffer[:end - self.size] = z_norm[tail:]
        self.ptr = end % self.size


# ─────────────────────────────────────────────────────────────────────────────
# 3. kNN HELPERS
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def compute_knn_indices(embeddings, k, batch_size=4096):
    """Compute top-k nearest neighbor indices using cosine similarity."""
    N = embeddings.size(0)
    emb = F.normalize(embeddings.to(device), p=2, dim=1)
    all_indices = []

    for i in range(0, N, batch_size):
        end = min(i + batch_size, N)
        sims = torch.matmul(emb[i:end], emb.T)
        for j in range(end - i):
            sims[j, i + j] = -1.0
        # Ensure k is not greater than the number of available elements
        effective_k = min(k, N - 1) if N > 1 else 0 # N-1 because we set self-similarity to -1
        if effective_k == 0:
            # Handle edge case where there's only one element or k is 0
            all_indices.append(torch.empty(end - i, 0, dtype=torch.long).cpu())
        else:
            all_indices.append(torch.topk(sims, k=effective_k, dim=1).indices.cpu())
        del sims
        torch.cuda.empty_cache()

    return torch.cat(all_indices, dim=0)


def knn_recall(gt_indices, pred_indices, k):
    """Recall@K: fraction of true neighbors recovered."""
    gt   = gt_indices.numpy()
    pred = pred_indices.numpy()
    total = 0.0
    # Adjusted to handle cases where k is effectively smaller than the original K
    # due to small N, to avoid division by zero if k becomes 0.
    divisor = k if k > 0 else 1 # Avoid division by zero

    for i in range(len(gt)):
        # Take only the first 'divisor' elements from gt[i] to compare against
        # This handles cases where gt_indices might have more than 'divisor' elements
        # but pred_indices (from topk) has 'divisor' elements.
        if pred.shape[1] > 0:
            intersection_size = len(np.intersect1d(gt[i, :divisor], pred[i], assume_unique=True))
            total += intersection_size / divisor
        # If pred has 0 elements (e.g., when N=1 and k is clamped to 0), contribution is 0
    return total / len(gt)


# ─────────────────────────────────────────────────────────────────────────────
# 4. PROFILING HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def get_storage_mb(tensor):
    """Disk/memory footprint of a float32 tensor in MB."""
    return tensor.element_size() * tensor.nelement() / (1024 ** 2)


def reset_gpu_stats():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)


def get_peak_gpu_mb():
    """Peak GPU memory allocated since last reset, in MB."""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    return 0.0


@torch.no_grad()
def benchmark_search_latency_ms(embeddings, k, n_queries=LATENCY_N_QUERIES):
    """
    Average per-query kNN search latency in milliseconds.
    Runs n_queries against the full database and averages wall time.
    """
    N = embeddings.size(0)
    db      = F.normalize(embeddings.to(device), p=2, dim=1)
    queries = db[:min(n_queries, N)] # Ensure n_queries doesn't exceed N

    if N <= 1 or k == 0: # Cannot perform kNN search for single element or k=0
        return 0.0

    # Warm-up pass (avoid CUDA JIT overhead)
    effective_k = min(k, N - 1) if N > 1 else 0
    if effective_k > 0:
        _ = torch.matmul(queries[:min(10, queries.size(0))], db.T)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()
        sims  = torch.matmul(queries, db.T)
        _     = torch.topk(sims, k=effective_k, dim=1).indices
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        return (elapsed / queries.size(0)) * 1000   # ms per query
    else:
        return 0.0


@torch.no_grad()
def benchmark_encode_latency_ms(model, embeddings, batch_size=256, n_queries=LATENCY_N_QUERIES):
    """
    Average per-sample encoding latency in milliseconds for a trained Encoder.
    Only meaningful for contrastive models (not PCA or native embeddings).
    """
    model.eval()
    N_total = embeddings.size(0)
    sample = embeddings[:min(n_queries, N_total)].to(device)

    if sample.size(0) == 0:
        return 0.0

    # Warm-up
    _ = model(sample[:min(16, sample.size(0))])
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()
    _ = model(sample)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    return (elapsed / sample.size(0)) * 1000   # ms per sample


# ─────────────────────────────────────────────────────────────────────────────
# 5. ADDITIONAL EVALUATION TASKS
# ─────────────────────────────────────────────────────────────────────────────

def cosine_sim_correlation(emb_base, emb_compressed,
                           sample_size=STS_SAMPLE_SIZE,
                           n_pairs=STS_N_PAIRS):
    """
    Task: Semantic Similarity Preservation (STS proxy)
    ───────────────────────────────────────────────────
    Sample random pairs, compute cosine similarity in both 768D and compressed
    space, then return the Spearman correlation.

    Interpretation: how faithfully does the compressed space preserve the
    relative similarity ordering of the original embeddings?
    A value of 1.0 = perfect ordering preservation.
    """
    N   = emb_base.size(0)
    if N < 2: # Need at least 2 samples to compute similarity
        return 0.0

    idx = torch.randperm(N)[:min(sample_size, N)]
    if idx.size(0) < 2: # Not enough samples after sampling
        return 0.0

    a = F.normalize(emb_base[idx].to(device),       p=2, dim=1)
    b = F.normalize(emb_compressed[idx].to(device), p=2, dim=1)

    # Random pairs (deduplicated)
    # Adjust n_pairs if the number of possible unique pairs is smaller
    max_possible_pairs = idx.size(0) * (idx.size(0) - 1) // 2
    actual_n_pairs = min(n_pairs, max_possible_pairs)

    if actual_n_pairs == 0:
        return 0.0

    i_idx = torch.randint(0, idx.size(0), (actual_n_pairs,))
    j_idx = torch.randint(0, idx.size(0), (actual_n_pairs,))
    mask  = i_idx != j_idx
    i_idx, j_idx = i_idx[mask], j_idx[mask]

    if i_idx.size(0) == 0: # No valid pairs after deduplication
        return 0.0

    sims_base = (a[i_idx] * a[j_idx]).sum(dim=1).cpu().numpy()
    sims_comp = (b[i_idx] * b[j_idx]).sum(dim=1).cpu().numpy()

    if len(sims_base) < 2: # spearmanr requires at least 2 data points
        return 0.0

    corr, _ = spearmanr(sims_base, sims_comp)
    return float(corr)


def clustering_nmi(base_cluster_labels, emb_compressed, n_clusters=N_CLUSTERS):
    """
    Task: Clustering Quality (Unsupervised label alignment)
    ───────────────────────────────────────────────────────
    KMeans is fit once on the 768D base embeddings.  We then cluster the
    compressed embeddings with the same K and compute the Normalised Mutual
    Information (NMI) between the two cluster assignment vectors.

    Interpretation: do the two spaces agree on which documents belong
    together?  NMI = 1.0 means perfect agreement; 0.0 means random.
    """
    if emb_compressed.size(0) < n_clusters: # KMeans requires N >= n_clusters
        return 0.0
    km     = MiniBatchKMeans(n_clusters=n_clusters, random_state=42,
                             n_init=3, batch_size=4096)
    labels = km.fit_predict(emb_compressed.numpy())
    return float(normalized_mutual_info_score(base_cluster_labels, labels))


def compute_mrr(gt_indices, emb_compressed, k=10, n_queries=500):
    """
    Task: Mean Reciprocal Rank (MRR) – simulates a re-ranking RAG scenario
    ───────────────────────────────────────────────────────────────────────
    For each query we treat the first GT neighbour as the 'relevant' document.
    We retrieve the top-k neighbours in compressed space and find the rank of
    that document.  MRR = mean(1 / rank).  If not in top-k, contribution = 0.

    Interpretation: how early does the most relevant document appear in the
    compressed retrieval list?  MRR = 1.0 means it's always rank-1.
    """
    N       = emb_compressed.size(0)
    n_q     = min(n_queries, N)

    if n_q == 0 or N <= 1: # Cannot compute MRR with no queries or single element
        return 0.0

    db      = F.normalize(emb_compressed.to(device), p=2, dim=1)
    queries = db[:n_q]

    with torch.no_grad():
        sims = torch.matmul(queries, db.T)
        for i in range(n_q):
            sims[i, i] = -1.0

        effective_k = min(k, N - 1) if N > 1 else 0
        if effective_k == 0: # No valid k for top_k
            top_k_idx = torch.empty(n_q, 0, dtype=torch.long).cpu().numpy()
        else:
            top_k_idx = torch.topk(sims, k=effective_k, dim=1).indices.cpu().numpy()

    gt_np = gt_indices[:n_q].numpy()
    rr    = []
    for i in range(n_q):
        # First GT neighbour is the "relevant" document
        # Ensure we don't try to access out of bounds for gt_np if N is small
        relevant_idx = 0 if gt_np.shape[1] > 0 else -1
        relevant = gt_np[i, relevant_idx] if relevant_idx != -1 else None

        if relevant is not None:
            rank_list = list(top_k_idx[i])
            if relevant in rank_list:
                rr.append(1.0 / (rank_list.index(relevant) + 1))
            else:
                rr.append(0.0)
        else: # No relevant GT neighbour found (e.g. N=1)
            rr.append(0.0)

    if len(rr) == 0:
        return 0.0
    return float(np.mean(rr))


# ─────────────────────────────────────────────────────────────────────────────
# 6. InfoNCE LOSS
# ─────────────────────────────────────────────────────────────────────────────
def simcse_info_nce(z_q, z_k, queue, tau):
    z_q_n = F.normalize(z_q, p=2, dim=1)
    z_k_n = F.normalize(z_k, p=2, dim=1)
    pos    = torch.sum(z_q_n * z_k_n, dim=1, keepdim=True) / tau
    neg    = torch.matmul(z_q_n, queue.T) / tau
    logits = torch.cat([pos, neg], dim=1)
    labels = torch.zeros(z_q.size(0), dtype=torch.long, device=z_q.device)
    return F.cross_entropy(logits, labels)


# ─────────────────────────────────────────────────────────────────────────────
# 7. ENCODE HELPER
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def encode_latent(model, embeddings, batch_size):
    model.eval()
    parts = []
    if len(embeddings) == 0: # Handle empty embeddings case
        return torch.empty(0, embeddings.size(1) if embeddings.dim() > 1 else 0)

    for i in range(0, len(embeddings), batch_size):
        batch = embeddings[i:i + batch_size].to(device)
        parts.append(model(batch).cpu())
    return torch.cat(parts, dim=0)


# ─────────────────────────────────────────────────────────────────────────────
# 8. TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
def train_simcse_moco(X, latent_dim, config):
    print(f"    Training SimCSE+MoCo  768D → {latent_dim}D")

    model_q = Encoder(
        input_dim    = config['input_dim'],
        latent_dim   = latent_dim,
        dropout_rate = config['dropout_rate'],
    ).to(device)
    if torch.cuda.device_count() > 1:
        model_q = nn.DataParallel(model_q)

    # Handle case where X is empty or too small for batching/training
    if X.size(0) < 2: # Need at least two samples for contrastive learning
        print("      Skipping training: not enough samples for SimCSE+MoCo.")
        return model_q # Return untrained model

    loader = DataLoader(
        TensorDataset(X),
        batch_size  = config['batch_size'],
        shuffle     = True,
        num_workers = 8,
        pin_memory  = True,
    )

    optimizer = optim.Adam(model_q.parameters(), lr=config['lr'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    model_k = copy.deepcopy(model_q).to(device)
    for p in model_k.parameters():
        p.requires_grad = False

    queue_size = config['moco_queue_size']
    # Ensure queue size is not larger than N, and at least batch_size
    if queue_size < config['batch_size']:
        queue_size = config['batch_size']
    if queue_size > X.size(0) * 2: # Arbitrary upper bound to avoid excessively large queues for small datasets
        queue_size = X.size(0) * 2

    queue = MoCoQueue(size=queue_size, dim=latent_dim, device=device)

    for epoch in range(config['epochs']):
        model_q.train()
        epoch_loss = 0.0
        for (X_batch,) in loader:
            X_batch = X_batch.to(device)
            if X_batch.size(0) < 2: # Skip if batch size is 1 for contrastive loss
                continue

            z_q = model_q(X_batch)
            with torch.no_grad():
                momentum_update(model_q, model_k, m=0.999)
                model_k.train()
                z_k = model_k(X_batch)

            loss = simcse_info_nce(z_q, z_k, queue.get(), config['tau'])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            queue.enqueue(F.normalize(z_k.detach(), p=2, dim=1))
            epoch_loss += loss.item()

        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"      Epoch {epoch+1:3d}/{config['epochs']} | Loss: {epoch_loss/len(loader):.4f}")

    return model_q


# ─────────────────────────────────────────────────────────────────────────────
# 9. UNIFIED EVALUATION FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_embeddings(emb, label, dim, gt_knn_per_k, bert_base,
                        base_cluster_labels, encoder_model=None):
    """
    Run all evaluation tasks and profiling for one set of embeddings.
    Returns a dict of metrics.

    Parameters
    ----------
    emb                : torch.Tensor  – embeddings to evaluate
    label              : str           – human-readable name
    dim                : int           – embedding dimensionality
    gt_knn_per_k       : dict[k -> Tensor] – GT neighbor indices for each k
    bert_base          : torch.Tensor  – original 768D embeddings
    base_cluster_labels: np.ndarray    – cluster labels from 768D KMeans
    encoder_model      : nn.Module | None – if contrastive, measure encode latency
    """
    result = {'method': label, 'dim': dim}

    # ── Storage ──────────────────────────────────────────────────────────────
    result['storage_mb'] = get_storage_mb(emb)

    # ── kNN Recall @ multiple k ──────────────────────────────────────────────
    for k_val in K_VALUES: # Loop through K_VALUES defined in config
        # Adjust k for `compute_knn_indices` if N is small
        effective_k = min(k_val, emb.size(0) - 1) if emb.size(0) > 1 else 0
        if effective_k == 0: # Cannot compute recall if k is 0 or N <= 1
            result[f'recall@{k_val}'] = 0.0
            result[f'gpu_knn_mb_k{k_val}'] = 0.0
            continue

        reset_gpu_stats()
        knn_pred = compute_knn_indices(emb, k=effective_k)
        # Ensure we are comparing with the correct GT for the effective_k
        gt_k = gt_knn_per_k[k_val] if k_val in gt_knn_per_k else None

        if gt_k is not None and knn_pred.size(0) > 0 and gt_k.size(0) == knn_pred.size(0):
             # Pass the original k_val to knn_recall to maintain consistent interpretation of 'K'
             result[f'recall@{k_val}'] = knn_recall(gt_k, knn_pred, k=effective_k)
        else:
             result[f'recall@{k_val}'] = 0.0
        result[f'gpu_knn_mb_k{k_val}'] = get_peak_gpu_mb()

    # ── Search Latency (use max k as worst-case, most realistic) ─────────────
    max_k = max(K_VALUES) if K_VALUES else 0
    result['search_latency_ms'] = benchmark_search_latency_ms(emb, k=max_k)

    # ── Encode Latency (contrastive models only) ─────────────────────────────
    if encoder_model is not None:
        result['encode_latency_ms'] = benchmark_encode_latency_ms(
            encoder_model, bert_base
        )
    else:
        result['encode_latency_ms'] = None   # N/A for PCA / native models

    # ── Task: Cosine Similarity Preservation (STS proxy) ────────────────────
    result['sts_spearman'] = cosine_sim_correlation(bert_base, emb)

    # ── Task: Clustering NMI ─────────────────────────────────────────────────
    result['cluster_nmi'] = clustering_nmi(base_cluster_labels, emb)

    # ── Task: Mean Reciprocal Rank (RAG simulation) ──────────────────────────
    mrr_k_val = 10 # MRR is fixed at k=10, but need to check if N is large enough
    # Ensure we use an effective k for MRR based on N
    effective_mrr_k = min(mrr_k_val, emb.size(0) - 1) if emb.size(0) > 1 else 0
    if effective_mrr_k == 0: # Cannot compute MRR if k is 0 or N <= 1
        result['mrr@10'] = 0.0
    else:
        # Ensure gt_knn_per_k has an entry for effective_mrr_k, or use closest available
        # For simplicity, we'll assume gt_knn_per_k[10] exists or handle gracefully.
        gt_for_mrr = gt_knn_per_k[mrr_k_val] if mrr_k_val in gt_knn_per_k else None

        if gt_for_mrr is not None:
            result['mrr@10'] = compute_mrr(gt_for_mrr, emb, k=effective_mrr_k)
        else:
            result['mrr@10'] = 0.0

    return result


# ─────────────────────────────────────────────────────────────────────────────
# 10. MAIN EXECUTION
# ─────────────────────────────────────────────────────────────────────────────

# ── Load BERT Base 768D ───────────────────────────────────────────────────────
print("Loading BERT Base 768D embeddings ...")
bert_base = torch.load(f'{OUTPUT_DIR}/embeddings_bert_base.pt',
                       weights_only=True).float()
N = len(bert_base)
print(f"  Shape: {bert_base.shape}")

# ── Ground truth kNN for EACH k (computed once from full 768D) ───────────────
print("\nComputing GT kNN (BERT Base 768D) for k =", K_VALUES, "...")
gt_knn_per_k = {}
for k in K_VALUES:
    # Ensure k for GT computation is valid for the dataset size N
    effective_k_gt = min(k, N - 1) if N > 1 else 0

    print(f"  k={k} (effective_k={effective_k_gt}) ...", end=" ", flush=True)
    if effective_k_gt > 0:
        gt_knn_per_k[k] = compute_knn_indices(bert_base, k=effective_k_gt)
        print("done")
    else:
        # For cases where effective_k is 0, provide an empty tensor or skip
        gt_knn_per_k[k] = torch.empty(N, 0, dtype=torch.long) # Create an empty tensor if no neighbors can be found
        print("skipped (N <= 1)")

# ── GT Clustering labels from 768D KMeans (fit once, reuse for NMI) ──────────
print(f"\nFitting KMeans (k={N_CLUSTERS}) on 768D for clustering NMI baseline ...")
# KMeans needs at least N_CLUSTERS samples
if N < N_CLUSTERS:
    print(f"  Skipping KMeans: Not enough samples (N={N}) for {N_CLUSTERS} clusters.")
    base_cluster_labels = np.zeros(N, dtype=int) # Default labels if clustering skipped
else:
    km_base = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=42,
                              n_init=3, batch_size=4096)
    base_cluster_labels = km_base.fit_predict(bert_base.numpy())
    print("  Done.")

results = []

# ─────────────────────────────────────────────────────────────────────────────
# PART A  —  Small BERT models
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("PART A : Small BERT models (native embeddings)")
print("=" * 70)

for model_name, dim in SMALL_MODELS:
    print(f"\n  [{model_name}]  {dim}D ...")
    emb = torch.load(f'{OUTPUT_DIR}/embeddings_{model_name}.pt',
                     weights_only=True).float()[:N]
    r = evaluate_embeddings(emb, model_name, dim, gt_knn_per_k,
                            bert_base, base_cluster_labels)
    results.append(r)
    for k_val in K_VALUES:
        print(f"    Recall@{k_val} = {r[f'recall@{k_val}']:.4f}")
    print(f"    STS Spearman = {r['sts_spearman']:.4f} | "
          f"Cluster NMI = {r['cluster_nmi']:.4f} | "
          f"MRR@10 = {r['mrr@10']:.4f}")
    print(f"    Storage = {r['storage_mb']:.1f} MB | "
          f"Search latency = {r['search_latency_ms']:.3f} ms/query")

# ─────────────────────────────────────────────────────────────────────────────
# PART B  —  PCA compression
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("PART B : PCA compression of BERT Base 768D")
print("=" * 70)

for latent_dim in COMPRESS_DIMS:
    print(f"\n  PCA  768D → {latent_dim}D ...")
    if N == 0: # Cannot perform PCA on empty data
        print("    Skipping PCA: No input embeddings.")
        continue
    if N == 1 and latent_dim > 0: # PCA with n_components > 0 and 1 sample gives error
        print("    Skipping PCA: Only one sample, cannot reduce dimensions effectively.")
        pca_emb = bert_base.clone() # Return original or empty if dim is different
    elif N > 1 and latent_dim >= N:
        print(f"    Warning: PCA components ({latent_dim}) >= number of samples ({N}). Adjusting to N-1.")
        pca = PCA(n_components=N - 1) # Reduce to N-1 if latent_dim is too large
        pca_emb = torch.tensor(pca.fit_transform(bert_base.numpy()),
                               dtype=torch.float32)
        # Pad with zeros if original latent_dim was larger than N-1, to match expected dimension
        if latent_dim > (N - 1):
            padding = torch.zeros(pca_emb.shape[0], latent_dim - (N - 1))
            pca_emb = torch.cat([pca_emb, padding], dim=1)
    else:
        pca = PCA(n_components=latent_dim)
        pca_emb = torch.tensor(pca.fit_transform(bert_base.numpy()),
                               dtype=torch.float32)

    if N > 1 and latent_dim < N: # Only print explained variance if PCA actually reduced dimensions
        print(f"    Explained variance: {pca.explained_variance_ratio_.sum():.4f}")

    r = evaluate_embeddings(pca_emb, f'PCA 768→{latent_dim}', latent_dim,
                            gt_knn_per_k, bert_base, base_cluster_labels)
    results.append(r)
    for k_val in K_VALUES:
        print(f"    Recall@{k_val} = {r[f'recall@{k_val}']:.4f}")
    print(f"    STS Spearman = {r['sts_spearman']:.4f} | "
          f"Cluster NMI = {r['cluster_nmi']:.4f} | "
          f"MRR@10 = {r['mrr@10']:.4f}")
    print(f"    Storage = {r['storage_mb']:.1f} MB | "
          f"Search latency = {r['search_latency_ms']:.3f} ms/query")

# ─────────────────────────────────────────────────────────────────────────────
# PART C  —  Contrastive compression (SimCSE + MoCo)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("PART C : Contrastive compression of BERT Base 768D")
print("=" * 70)

for latent_dim in COMPRESS_DIMS:
    print(f"\n  Contrastive 768D → {latent_dim}D ...")
    model = train_simcse_moco(bert_base, latent_dim, CONFIG)
    compressed_emb = encode_latent(model, bert_base, batch_size=CONFIG['batch_size'])

    r = evaluate_embeddings(compressed_emb, f'Contrastive 768→{latent_dim}',
                            latent_dim, gt_knn_per_k, bert_base,
                            base_cluster_labels, encoder_model=model)
    results.append(r)
    for k_val in K_VALUES:
        print(f"    Recall@{k_val} = {r[f'recall@{k_val}']:.4f}")
    print(f"    STS Spearman = {r['sts_spearman']:.4f} | "
          f"Cluster NMI = {r['cluster_nmi']:.4f} | "
          f"MRR@10 = {r['mrr@10']:.4f}")
    print(f"    Storage = {r['storage_mb']:.1f} MB | "
          f"Search latency = {r['search_latency_ms']:.3f} ms/query | "
          f"Encode latency = {r['encode_latency_ms']:.3f} ms/sample")

    del model
    torch.cuda.empty_cache()

# ─────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY TABLE — All methods, all metrics
# ─────────────────────────────────────────────────────────────────────────────
recall_cols = " | ".join(f"{'R@'+str(k):<8}" for k in K_VALUES)
header = (f"{'Method':<28} | {'Dim':<5} | {recall_cols} | "
          f"{'STS-ρ':<7} | {'NMI':<6} | {'MRR@10':<7} | "
          f"{'Store(MB)':<10} | {'Search(ms)':<11} | {'Encode(ms)':<10}")

print("\n" + "=" * len(header))
print("FULL RESULTS")
print("=" * len(header))
print(header)
print("-" * len(header))
for r in results:
    rec_str = " | ".join(f"{r[f'recall@{k_val}']:<8.4f}" for k_val in K_VALUES)
    enc = f"{r['encode_latency_ms']:.3f}" if r['encode_latency_ms'] is not None else "N/A"
    print(
        f"{r['method']:<28} | {r['dim']:<5} | {rec_str} | "
        f"{r['sts_spearman']:<7.4f} | {r['cluster_nmi']:<6.4f} | "
        f"{r['mrr@10']:<7.4f} | {r['storage_mb']:<10.1f} | "
        f"{r['search_latency_ms']:<11.3f} | {enc:<10}"
    )
print("=" * len(header))

# ─────────────────────────────────────────────────────────────────────────────
# 768D vs 128D  — head-to-head RAG cost/quality tradeoff summary
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("768D vs 128D : Cost / Quality Tradeoff")
print("=" * 60)

# Baseline: use the 768D numbers directly
storage_768  = get_storage_mb(bert_base)
# Ensure k is valid for benchmark_search_latency_ms as well
search_k_768 = 10
if N <= 1:
    search_768 = 0.0
else:
    search_768   = benchmark_search_latency_ms(bert_base, k=min(search_k_768, N-1))

# recall_768 = {k: 1.0 for k in K_VALUES}   # GT is itself; recall = 1 by definition (only if effective_k > 0)
recall_768 = {}
for k_val in K_VALUES:
    recall_768[k_val] = 1.0 if (N > 1 and k_val <= N - 1) else 0.0 # Only 1.0 if meaningful

# Best 128D result (prefer contrastive over PCA if available)
cand_128 = [r for r in results if r['dim'] == 128]
best_128  = max(cand_128, key=lambda r: r.get('recall@1', 0.0)) if cand_128 else None # Use recall@1 for sorting, or 0 if not present

print(f"\n{'Metric':<25} | {'768D (baseline)':<18} | {'128D best':<18} | {'Δ / ratio'}")
print("-" * 75)
if best_128:
    rows = [
        ("Storage (MB)",
         f"{storage_768:.1f}",
         f"{best_128['storage_mb']:.1f}",
         f"{storage_768/best_128['storage_mb']:.1f}× smaller" if best_128['storage_mb'] > 0 else "N/A"),
        ("Search latency (ms)",
         f"{search_768:.3f}",
         f"{best_128['search_latency_ms']:.3f}",
         f"{search_768/best_128['search_latency_ms']:.1f}× faster" if best_128['search_latency_ms'] > 0 else "N/A"),
    ]
    for k_val in K_VALUES:
        # Ensure we don't try to calculate percentage change if recall is 0
        recall_drop_str = f"{(1 - best_128[f'recall@{k_val}'])*100:.1f}% quality drop" if recall_768[k_val] > 0 else "N/A"
        rows.append((
            f"Recall@{k_val}",
            f"{recall_768[k_val]:.4f}",
            f"{best_128[f'recall@{k_val}']:.4f}",
            recall_drop_str
        ))
    # Only add STS Spearman and MRR if N > 1 for meaningful calculation
    if N > 1:
        rows += [
            ("STS Spearman",
             "1.0000",
             f"{best_128['sts_spearman']:.4f}",
             f"{(1 - best_128['sts_spearman'])*100:.1f}% drop" if best_128['sts_spearman'] > 0 else "N/A"),
            ("MRR@10",
             "1.0000",
             f"{best_128['mrr@10']:.4f}",
             f"{(1 - best_128['mrr@10'])*100:.1f}% drop" if best_128['mrr@10'] > 0 else "N/A"),
        ]

    # Only add GPU memory if there was GPU usage
    gpu_knn_768_mb = get_peak_gpu_mb()
    if gpu_knn_768_mb > 0 or best_128.get('gpu_knn_mb_k' + str(max(K_VALUES) if K_VALUES else 0), 0) > 0: # Check if either has GPU usage
        rows.append((
            "GPU mem – kNN (MB)",
            f"{gpu_knn_768_mb:.0f}",
            f"{best_128.get('gpu_knn_mb_k' + str(max(K_VALUES) if K_VALUES else 0), 0):.0f}", # Use max K for 128D as well
            f"{best_128.get('gpu_knn_mb_k' + str(max(K_VALUES) if K_VALUES else 0), 0)/max(gpu_knn_768_mb,1):.2f}x of 768D" if gpu_knn_768_mb > 0 else "N/A"
        ))

    for metric, v768, v128, delta in rows:
        print(f"{metric:<25} | {v768:<18} | {v128:<18} | {delta}")

print("=" * 60)
print(f"\n  Best 128D method : {best_128['method'] if best_128 else 'N/A'}")
print("=" * 60)

Loading BERT Base 768D embeddings ...
  Shape: torch.Size([4, 768])

Computing GT kNN (BERT Base 768D) for k = [1, 2, 3] ...
  k=1 (effective_k=1) ... done
  k=2 (effective_k=2) ... done
  k=3 (effective_k=3) ... done

Fitting KMeans (k=20) on 768D for clustering NMI baseline ...
  Skipping KMeans: Not enough samples (N=4) for 20 clusters.

PART A : Small BERT models (native embeddings)

  [bert_tiny]  128D ...
    Recall@1 = 0.5000
    Recall@2 = 0.5000
    Recall@3 = 1.0000
    STS Spearman = 0.5789 | Cluster NMI = 0.0000 | MRR@10 = 0.0000
    Storage = 0.0 MB | Search latency = 0.044 ms/query

  [bert_mini]  256D ...
    Recall@1 = 0.5000
    Recall@2 = 0.7500
    Recall@3 = 1.0000
    STS Spearman = 0.2500 | Cluster NMI = 0.0000 | MRR@10 = 0.0000
    Storage = 0.0 MB | Search latency = 0.074 ms/query

  [bert_small]  512D ...
    Recall@1 = 0.5000
    Recall@2 = 0.7500
    Recall@3 = 1.0000
    STS Spearman = 0.7778 | Cluster NMI = 0.0000 | MRR@10 = 0.0000
    Storage = 0.0 MB | Se

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


      Epoch  10/40 | Loss: 1.6343
      Epoch  20/40 | Loss: 1.4570
      Epoch  30/40 | Loss: 1.2039
      Epoch  40/40 | Loss: 1.3387
    Recall@1 = 0.7500
    Recall@2 = 0.8750
    Recall@3 = 1.0000
    STS Spearman = 1.0000 | Cluster NMI = 0.0000 | MRR@10 = 0.0000
    Storage = 0.0 MB | Search latency = 0.027 ms/query | Encode latency = 0.079 ms/sample

  Contrastive 768D → 256D ...
    Training SimCSE+MoCo  768D → 256D
      Epoch  10/40 | Loss: 1.3508
      Epoch  20/40 | Loss: 1.2359
      Epoch  30/40 | Loss: 1.0563
      Epoch  40/40 | Loss: 1.0318
    Recall@1 = 0.7500
    Recall@2 = 0.8750
    Recall@3 = 1.0000
    STS Spearman = 0.8182 | Cluster NMI = 0.0000 | MRR@10 = 0.0000
    Storage = 0.0 MB | Search latency = 0.023 ms/query | Encode latency = 0.045 ms/sample

  Contrastive 768D → 512D ...
    Training SimCSE+MoCo  768D → 512D
      Epoch  10/40 | Loss: 1.3485
      Epoch  20/40 | Loss: 1.1713
      Epoch  30/40 | Loss: 1.0227
      Epoch  40/40 | Loss: 1.2981
    Reca